In [1]:
from pydrake.all import (
    StartMeshcat,
    AddMultibodyPlantSceneGraph,
    Parser,
    AddDefaultVisualization,
    DiagramBuilder,
    PointCloud,
    Fields,
    BaseField,
    RigidTransform,
    RotationMatrix,
    InverseKinematics,
    Solve,
)
from pathlib import Path
import numpy as np
import open3d as o3d
from scipy.spatial.transform import Rotation

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [3]:
project_dir = Path("/home/noor/so101-drake")

In [4]:
builder = DiagramBuilder()

plant, scene_graph = AddMultibodyPlantSceneGraph(
    builder,
    time_step=0.001,
)

parser = Parser(plant)
parser.AddModels(project_dir / "models" / "objects" / "surface.sdf")
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("surface_link")
)

T_world_base = np.eye(4)
rotvec = np.array([0, 0, 1]) * np.pi/2
T_world_base[:3, :3] = Rotation.from_rotvec(rotvec).as_matrix()
T_world_base[:3, 3] = [0, -0.1775, 0.0074]

so101 = parser.AddModels(
    project_dir / "models" / "SO101" / "so101_new_calib_drake_hydro.urdf"
)[0]
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link"),
    RigidTransform(
        RotationMatrix(T_world_base[:3, :3]), 
        T_world_base[:3, 3]
    )
)

# grasp = np.array([
#     [ 0.20856473, -0.96235112, -0.17430128, -0.04784185],
#     [-0.97346751, -0.22142707,  0.05771393,  0.04177497],
#     [-0.09413604,  0.15763952, -0.98299962,  0.15351727],
#     [ 0.        ,  0.        ,  0.        ,  1.        ],
# ])
# grasp = np.eye(4)
# grasp[:3, :3] = Rotation.from_rotvec([0, 0, np.pi/2]).as_matrix()
# grasp[:3, 3] = [0.13, 0.02, 0.2]

# g = parser.AddModels("/home/noor/SO-ARM100/Simulation/SO101/so101_gripper.urdf")[0]
# plant.WeldFrames(
#     plant.world_frame(), 
#     plant.GetFrameByName("gripper_link", g), 
#     RigidTransform(
#         # RotationMatrix.MakeYRotation(np.pi) @ RotationMatrix(grasp[:3, :3]),
#         RotationMatrix(grasp[:3, :3]),
#         # [grasp[0, 3], grasp[1, 3]+0.015, grasp[2, 3]]
#         grasp[:3, 3]
#     )
# )

plant.Finalize()

AddDefaultVisualization(builder, meshcat)

diagram = builder.Build()
context = diagram.CreateDefaultContext()

q_init = np.zeros(6)
q_rest = np.array([0, -1.822, 1.55, 0.906, 0, 0])
plant.SetPositions(plant.GetMyMutableContextFromRoot(context), q_rest)

diagram.ForcedPublish(context)

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html



In [5]:
filtered = o3d.io.read_point_cloud(project_dir / "assets" / "filtered.ply")

points = np.asarray(filtered.points)
colors = np.asarray(filtered.colors)  # sRGB in [0, 1] from Open3D

def srgb_to_linear(c):
    c = np.clip(c, 0.0, 1.0)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

colors_linear = srgb_to_linear(colors)

# Round and clip before casting to avoid uint8 wraparound.
rgbs_uint8 = np.clip(np.round(255.0 * colors_linear), 0, 255).astype(np.uint8)
rgbs_uint8 = np.flip(rgbs_uint8, axis=1)

pc = PointCloud(
    new_size=len(points),
    fields=Fields(BaseField.kXYZs | BaseField.kRGBs),
)
pc.mutable_xyzs()[:] = points.T
pc.mutable_rgbs()[:] = rgbs_uint8.T

meshcat.SetObject("/my_surface/rgb_points", pc, point_size=0.003)

In [11]:
def solve_ik_place():
    grasp = np.eye(4)
    grasp[:3, :3] = Rotation.from_rotvec([0, 0, np.pi/2]).as_matrix()
    grasp[:3, 3] = [0.13, 0.0, 0.2]

    # initial guess
    ik = InverseKinematics(plant)
    ik.get_mutable_prog().AddQuadraticErrorCost(1.0, q_init, ik.q())
    ik.get_mutable_prog().SetInitialGuess(ik.q(), q_init)

    # constraints
    ik.AddPositionConstraint(
        plant.GetFrameByName("gripper_link", so101),
        [0.0, 0.0, 0.0],
        plant.world_frame(),
        grasp[:3, 3],
        grasp[:3, 3]
    )
    ik.AddOrientationConstraint(
        plant.GetFrameByName("gripper_link", so101),
        RotationMatrix(),
        plant.world_frame(),
        RotationMatrix(grasp[:3, :3]),
        np.pi/16
    )
    ik.get_mutable_prog().AddBoundingBoxConstraint(
        np.pi/4, np.pi/4, ik.q()[5]
    )

    # solve
    result = Solve(ik.prog())
    if result.is_success():
        plant.SetPositions(
            plant.GetMyMutableContextFromRoot(context), 
            result.GetSolution(ik.q())
        )
        diagram.ForcedPublish(context)
        return result.GetSolution(ik.q())
    else:
        return None


print(solve_ik_place())

[ 0.75222731 -0.1107525  -0.007926    1.50925026  0.72288082  0.78539816]


In [12]:
def solve_ik_pick(grasp):
    # initial guess
    ik = InverseKinematics(plant)
    ik.get_mutable_prog().AddQuadraticErrorCost(1.0, q_init, ik.q())
    ik.get_mutable_prog().SetInitialGuess(ik.q(), q_init)

    # constraints
    ik.AddPositionConstraint(
        plant.GetFrameByName("gripper_link", so101),
        [0.0, 0.0, 0.0],
        plant.world_frame(),
        grasp[:3, 3] + np.array([-0.015, 0.01, 0]),
        grasp[:3, 3] + np.array([-0.015, 0.01, 0])
    )
    ik.AddOrientationConstraint(
        plant.GetFrameByName("gripper_link", so101),
        RotationMatrix(),
        plant.world_frame(),
        RotationMatrix.MakeYRotation(np.pi) @ RotationMatrix(grasp[:3, :3]),
        np.pi/16
    )
    ik.get_mutable_prog().AddBoundingBoxConstraint(
        np.pi/4, np.pi/4, ik.q()[5]
    )

    # solve
    result = Solve(ik.prog())
    if result.is_success():
        plant.SetPositions(
            plant.GetMyMutableContextFromRoot(context), 
            result.GetSolution(ik.q())
        )
        diagram.ForcedPublish(context)
        return result.GetSolution(ik.q())
    else:
        return None


grasp = np.array([
    [ 0.20856473, -0.96235112, -0.17430128, -0.04784185],
    [-0.97346751, -0.22142707,  0.05771393,  0.04177497],
    [-0.09413604,  0.15763952, -0.98299962,  0.15351727],
    [ 0.        ,  0.        ,  0.        ,  1.        ],
])
print(solve_ik_pick(grasp))

[-0.31961414  0.05712364  0.13169853  1.34462259  2.59307222  0.78539816]
